In [ ]:
import java.io.File;
import java.io.IOException;
import java.nio.channels.FileChannel;
import java.nio.file.StandardOpenOption;

import dev.chpg.pg.api.AttributeValue;
import dev.chpg.pg.global.GlobalGraph;
import dev.chpg.pg.io.DirectGraphBufferReader;

In [ ]:
long start = System.currentTimeMillis();
File file = new File(new File("data"), "xinu.dgb");
GlobalGraph targetGraph = new GlobalGraph();
try (FileChannel channel = FileChannel.open(file.toPath(), StandardOpenOption.READ)) {
    DirectGraphBufferReader.read(channel, targetGraph, targetGraph.factory(), targetGraph.factory());
}
long stop = System.currentTimeMillis();

System.out.println("Time: " + (stop-start));
System.out.println("Nodes: " + targetGraph.nodes().size());
System.out.println("Edges: " + targetGraph.edges().size());

System.out.println(
    targetGraph.nodes()
        .withAnyTag("XCSG.Function")
        .withAttribute("XCSG.name", AttributeValue.value("freebuf"))
        .toString()
);

System.out.println(
    targetGraph.nodes()
        .withAnyTag("XCSG.Function")
        .withAttribute("XCSG.name", AttributeValue.value("freebuf"))
        .one().get().tags().toString()
);

System.out.println(
    targetGraph.edges()
        .withAnyTag("XCSG.Call")
        .size()
);

In [ ]:
import java.io.File;
import java.io.IOException;
import java.nio.channels.FileChannel;
import java.nio.file.StandardOpenOption;

import dev.chpg.pg.api.AttributeValue;
import dev.chpg.pg.io.DirectGraphBufferReader;
import dev.chpg.pg.multiverse.ephemeral.EphemeralGraph;
import dev.chpg.pg.multiverse.universe.Universe;
import dev.chpg.pg.multiverse.universe.UniverseGraph;

long start = System.currentTimeMillis();
File file = new File(new File("data"), "xinu.dgb");
Universe universe = new Universe();

// 2. Create the ingestion transaction
EphemeralGraph ingestionSandbox = new EphemeralGraph(universe);

// 3. Stream directly into the transaction!
// EphemeralGraph safely acts as the Graph, NodeFactory, and EdgeFactory.
try (FileChannel channel = FileChannel.open(file.toPath(), StandardOpenOption.READ)) {
    DirectGraphBufferReader.read(channel, ingestionSandbox, ingestionSandbox, ingestionSandbox);
}

// 4. Compile the transaction into the columnar baseline
UniverseGraph activeBaseline = universe.promote(ingestionSandbox);

// 5. Spin up a fresh sandbox for analysis
EphemeralGraph sandbox = new EphemeralGraph(universe);
long stop = System.currentTimeMillis();

System.out.println("Time: " + (stop-start));
System.out.println("Nodes: " + targetGraph.nodes().size());
System.out.println("Edges: " + targetGraph.edges().size());

System.out.println(
    targetGraph.nodes()
        .withAnyTag("XCSG.Function")
        .withAttribute("XCSG.name", AttributeValue.value("freebuf"))
        .toString()
);

System.out.println(
    targetGraph.nodes()
        .withAnyTag("XCSG.Function")
        .withAttribute("XCSG.name", AttributeValue.value("freebuf"))
        .one().get().tags().toString()
);

System.out.println(
    targetGraph.edges()
        .withAnyTag("XCSG.Call")
        .size()
);


In [ ]:
import java.nio.file.Files;
import java.nio.file.Paths;
import java.nio.charset.StandardCharsets;
import java.util.Base64;
import dev.chpg.pg.api.AttributeValue;
import dev.chpg.pg.multiverse.universe.*;
import dev.pgv.exporter.*; 
import java.io.ByteArrayOutputStream;
import java.util.List;
import java.util.Map;

// 1. Create the backend Universe graph
Universe universe = new Universe();
int nodeId = universe.idGenerator().createNodeId();
UniverseNode helloNode = new UniverseNode(universe, nodeId);

// 2. Adapter
record GraphNode(String id, List<String> tags, Map<String, Object> attributes) implements ExportNode {}
record GraphEdge(String id, String source, String target, List<String> tags, Map<String, Object> attributes) implements ExportEdge {}
record GraphSnapshot(String graphId, long version, List<GraphNode> nodesList, List<GraphEdge> edgesList) implements ExportGraph {
    public ExportSchema schema() { return null; }
    public Iterable<? extends ExportNode> nodes() { return nodesList; }
    public Iterable<? extends ExportEdge> edges() { return edgesList; }
}

GraphNode exportHelloNode = new GraphNode(
    String.valueOf(helloNode.id()), 
    List.of("TestNode"), 
    Map.of("name", "hello") 
);

GraphSnapshot snapshot = new GraphSnapshot("hello-world-graph", 1, List.of(exportHelloNode), List.of());

// 3. Serialize JSON
ByteArrayOutputStream baos = new ByteArrayOutputStream();
PgvExporter exporter = new PgvExporter();
exporter.exportGraph(snapshot, baos);
String jsonPayload = baos.toString(StandardCharsets.UTF_8);

// 4. Read the Vite Bundles
String jsBundle = Files.readString(Paths.get("/app/pgv/dist/pgv-bundle.js"));
jsBundle = jsBundle.replace("</script>", "<\\/script>");

// Grab the CSS file sitting in your dist folder!
String cssBundle = Files.readString(Paths.get("/app/pgv/dist/graph-core.css"));

// 5. Build a completely isolated HTML document
String iframeHtml = """
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="utf-8">
        <style>
            body { margin: 0; padding: 0; overflow: hidden; font-family: sans-serif; background: #fafafa; }
            /* INJECT THE CSS BUNDLE HERE */
            __CSS_BUNDLE__
        </style>
    </head>
    <body>
        <div id="pgv-viz" style="width: 100vw; height: 100vh;"></div>
        <script>
            __JS_BUNDLE__
        </script>
        <script>
            try {
                let payload = __JSON_PAYLOAD__;
                
                payload.schema = { 
                    nodes: { "TestNode": { color: "#4f46e5", label: "Test Node" } }, 
                    edges: {} 
                };
                
                const container = document.getElementById("pgv-viz");
                const graphSnapshot = pgv.createGraphSnapshot(payload);
                const view = new pgv.GraphView(container, payload.schema, {
                    layoutOptions: { nodeWidth: 240, nodeHeight: 94, layerSpacing: 152, nodeSpacing: 290, margin: 36 },
                    usePanZoom: true,
                    useThemeToggle: true
                });
                
                view.setGraph(graphSnapshot);
                
            } catch (err) {
                document.getElementById("pgv-viz").innerHTML = "<div style='color:red; padding: 20px;'>" + 
                    "<h3>Visualizer Crash:</h3>" + err.message + "<br><br><pre>" + err.stack + "</pre></div>";
            }
        </script>
    </body>
    </html>
    """
    .replace("__CSS_BUNDLE__", cssBundle)
    .replace("__JS_BUNDLE__", jsBundle)
    .replace("__JSON_PAYLOAD__", jsonPayload);

// 6. Encode the entire webpage to Base64
String base64Html = Base64.getEncoder().encodeToString(iframeHtml.getBytes(StandardCharsets.UTF_8));

// 7. Inject into a sandboxed iframe
String finalOutput = """
    <iframe src="data:text/html;base64,__BASE64_HTML__" width="100%" height="600px" style="border: 1px solid #ccc; border-radius: 8px; resize: vertical;"></iframe>
    """.replace("__BASE64_HTML__", base64Html);

display(finalOutput, "text/html");

In [ ]:
import java.nio.file.Files;
import java.nio.file.Paths;
import java.nio.charset.StandardCharsets;
import dev.chpg.pg.api.AttributeValue;
import dev.chpg.pg.multiverse.universe.*;
import dev.pgv.exporter.*; 
import java.io.ByteArrayOutputStream;
import java.util.List;
import java.util.Map;

// 1. Create the backend Universe graph
Universe universe = new Universe();
int nodeId = universe.idGenerator().createNodeId();
UniverseNode helloNode = new UniverseNode(universe, nodeId);

// 2. Adapter
record GraphNode(String id, List<String> tags, Map<String, Object> attributes) implements ExportNode {}
record GraphEdge(String id, String source, String target, List<String> tags, Map<String, Object> attributes) implements ExportEdge {}
record GraphSnapshot(String graphId, long version, List<GraphNode> nodesList, List<GraphEdge> edgesList) implements ExportGraph {
    public ExportSchema schema() { return null; }
    public Iterable<? extends ExportNode> nodes() { return nodesList; }
    public Iterable<? extends ExportEdge> edges() { return edgesList; }
}

GraphNode exportHelloNode = new GraphNode(
    String.valueOf(helloNode.id()), 
    List.of("TestNode"), 
    Map.of("name", "hello") 
);

GraphSnapshot snapshot = new GraphSnapshot("hello-world-graph", 1, List.of(exportHelloNode), List.of());

// 3. Serialize JSON
ByteArrayOutputStream baos = new ByteArrayOutputStream();
PgvExporter exporter = new PgvExporter();
exporter.exportGraph(snapshot, baos);
String jsonPayload = baos.toString(StandardCharsets.UTF_8);

// 4. Read the Vite Bundles and escape closing script tags
String jsBundle = Files.readString(Paths.get("/app/pgv/dist/pgv-bundle.js"));
jsBundle = jsBundle.replace("</script>", "<\\/script>");

String cssBundle = Files.readString(Paths.get("/app/pgv/dist/graph-core.css"));

// Generate a unique container ID so multiple cells don't clash
String containerId = "pgv-viz-" + System.currentTimeMillis();

// 5. Build the direct HTML injection string using safe .replace() tokens
String directHtml = """
    <style>
        __CSS_BUNDLE__
    </style>
    <div id="__CONTAINER_ID__" style="width: 100%; height: 600px; border: 1px solid #ccc; border-radius: 8px; background: #fafafa; resize: vertical; overflow: hidden;"></div>
    <script>
        // Evaluate the IIFE bundle directly in the notebook context
        __JS_BUNDLE__
    </script>
    <script>
        try {
            let payload = __JSON_PAYLOAD__;
            
            payload.schema = { 
                nodes: { "TestNode": { color: "#4f46e5", label: "Test Node" } }, 
                edges: {} 
            };
            
            const container = document.getElementById("__CONTAINER_ID__");
            const graphSnapshot = pgv.createGraphSnapshot(payload);
            const view = new pgv.GraphView(container, payload.schema, {
                layoutOptions: { nodeWidth: 240, nodeHeight: 94, layerSpacing: 152, nodeSpacing: 290, margin: 36 },
                usePanZoom: true,
                useThemeToggle: true
            });
            
            view.setGraph(graphSnapshot);
        } catch (err) {
            document.getElementById("__CONTAINER_ID__").innerHTML = "<div style='color:red; padding: 20px;'>" + 
                "<h3>Visualizer Crash:</h3>" + err.message + "<br><br><pre>" + err.stack + "</pre></div>";
        }
    </script>
    """
    .replace("__CSS_BUNDLE__", cssBundle)
    .replace("__CONTAINER_ID__", containerId)
    .replace("__JS_BUNDLE__", jsBundle)
    .replace("__JSON_PAYLOAD__", jsonPayload);

display(directHtml, "text/html");